# Series 2 — Post 3: Stream-Stream Joins

## Objective

Join two continuously changing event streams:

- Order events
- Payment events

The implementation will cover:

- Watermarks on both streams
- Event-time join constraints
- Inner stream-stream joins
- Left outer stream-stream joins
- Late and unmatched records
- State-store behavior
- Append output mode

## Core question

How long should Spark retain unmatched records while waiting for the corresponding event from the other stream?

In [0]:
%sql
--Create schema and volume
CREATE SCHEMA IF NOT EXISTS workspace.stream_join_lab;

CREATE VOLUME IF NOT EXISTS
workspace.stream_join_lab.stream_join_volume;

USE CATALOG workspace;
USE SCHEMA stream_join_lab;

In [0]:
#Define Paths and Objects
catalog = "workspace"
schema = "stream_join_lab"

base_path = (
    "/Volumes/workspace/stream_join_lab/"
    "stream_join_volume"
)

orders_source_path = f"{base_path}/orders"
payments_source_path = f"{base_path}/payments"

checkpoint_base = f"{base_path}/checkpoints"

inner_join_table = (
    "workspace.stream_join_lab."
    "order_payment_inner_join"
)

inner_join_checkpoint = (
    f"{checkpoint_base}/order_payment_inner_join"
)

print(f"Orders source: {orders_source_path}")
print(f"Payments source: {payments_source_path}")
print(f"Checkpoint base: {checkpoint_base}")

In [0]:
#Import the required types
from datetime import datetime

from pyspark.sql import Row
from pyspark.sql import functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType,
)

In [0]:
#Define Schema
order_schema = StructType([
    StructField("order_event_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("order_amount", DoubleType(), False),
    StructField("order_status", StringType(), False),
    StructField("order_ts", TimestampType(), False),
])

In [0]:
#Define Schema
payment_schema = StructType([
    StructField("payment_event_id", StringType(), False),
    StructField("payment_id", StringType(), False),
    StructField("order_id", StringType(), False),
    StructField("payment_amount", DoubleType(), False),
    StructField("payment_status", StringType(), False),
    StructField("payment_ts", TimestampType(), False),
])

In [0]:
#Reset the new volume
for active_query in spark.streams.active:
    active_query.stop()

spark.sql(
    f"DROP TABLE IF EXISTS {inner_join_table}"
)

dbutils.fs.rm(orders_source_path, True)
dbutils.fs.rm(payments_source_path, True)
dbutils.fs.rm(checkpoint_base, True)

dbutils.fs.mkdirs(orders_source_path)
dbutils.fs.mkdirs(payments_source_path)
dbutils.fs.mkdirs(checkpoint_base)

print("✅ Post 3 lab reset")

In [0]:
#Create batch 1
orders_batch_1 = [
    Row(
        order_event_id="OE-101",
        order_id="ORD-101",
        user_id="U-101",
        order_amount=100.0,
        order_status="CREATED",
        order_ts=datetime(2026, 7, 28, 10, 0, 0),
    ),
    Row(
        order_event_id="OE-102",
        order_id="ORD-102",
        user_id="U-102",
        order_amount=80.0,
        order_status="CREATED",
        order_ts=datetime(2026, 7, 28, 10, 3, 0),
    ),
    Row(
        order_event_id="OE-103",
        order_id="ORD-103",
        user_id="U-103",
        order_amount=120.0,
        order_status="CREATED",
        order_ts=datetime(2026, 7, 28, 10, 8, 0),
    ),
]

In [0]:
payments_batch_1 = [
    Row(
        payment_event_id="PE-101",
        payment_id="PAY-101",
        order_id="ORD-101",
        payment_amount=100.0,
        payment_status="SUCCESS",
        payment_ts=datetime(2026, 7, 28, 10, 2, 0),
    ),
    Row(
        payment_event_id="PE-103",
        payment_id="PAY-103",
        order_id="ORD-103",
        payment_amount=120.0,
        payment_status="SUCCESS",
        payment_ts=datetime(2026, 7, 28, 10, 9, 0),
    ),
]

In [0]:
#Write Batch 1 files
orders_batch_1_df = spark.createDataFrame(
    orders_batch_1,
    schema=order_schema,
)

payments_batch_1_df = spark.createDataFrame(
    payments_batch_1,
    schema=payment_schema,
)

In [0]:
(
    orders_batch_1_df
    .coalesce(1)
    .write
    .mode("append")
    .json(f"{orders_source_path}/batch_1")
)

(
    payments_batch_1_df
    .coalesce(1)
    .write
    .mode("append")
    .json(f"{payments_source_path}/batch_1")
)

print("✅ Orders Batch 1 written")
print("✅ Payments Batch 1 written")

In [0]:
display(
    orders_batch_1_df.orderBy("order_ts")
)

display(
    payments_batch_1_df.orderBy("payment_ts")
)

In [0]:
#Create streaming reads
orders_stream_df = (
    spark.readStream
    .schema(order_schema)
    .option("recursiveFileLookup", "true")
    .json(orders_source_path)
)

In [0]:
payments_stream_df = (
    spark.readStream
    .schema(payment_schema)
    .option("recursiveFileLookup", "true")
    .json(payments_source_path)
)

In [0]:
print(
    f"Orders streaming: "
    f"{orders_stream_df.isStreaming}"
)

print(
    f"Payments streaming: "
    f"{payments_stream_df.isStreaming}"
)

# Stream-Static Join

## Objective

Enrich a continuously arriving order stream with customer attributes stored in a static Delta table.

The flow is:

Streaming orders  
→ Join with static customer dimension  
→ Add customer segment, region, and risk level  
→ Preserve unmatched orders with an unknown-member pattern

## Why this join is simpler

Only the order side is streaming.

The customer dimension is read as a regular static DataFrame, so Spark does not need to maintain join state for both sides.

That means:

- No watermark is required on the static side
- No event-time range condition is required
- The join behaves like enriching each incoming micro-batch with reference data
- A left join prevents unmatched orders from being dropped

## Business use case

This pattern is useful when a streaming event needs descriptive context from relatively stable reference data, such as:

- Orders joined to customer segments
- Transactions joined to merchant details
- Machine events joined to an asset registry
- Shipments joined to warehouse locations

## Expected behavior

Known customers are enriched with their customer attributes.

Unknown customers are preserved and mapped to:

- `Unknown Customer`
- `UNKNOWN` segment
- `UNKNOWN` region
- `UNKNOWN_CUSTOMER` match status

This allows downstream systems to retain the order while clearly identifying missing reference data.

In [0]:
#Create a static customer dimension
customer_dimension = [
    Row(
        user_id="U-101",
        customer_name="Ava Johnson",
        customer_segment="GOLD",
        region="WEST",
        risk_level="LOW",
    ),
    Row(
        user_id="U-102",
        customer_name="Noah Williams",
        customer_segment="SILVER",
        region="EAST",
        risk_level="MEDIUM",
    ),
    Row(
        user_id="U-103",
        customer_name="Emma Brown",
        customer_segment="PLATINUM",
        region="CENTRAL",
        risk_level="LOW",
    ),
]

In [0]:
customer_dimension_df = spark.createDataFrame(
    customer_dimension
)

customer_dimension_table = (
    "workspace.stream_join_lab.customer_dimension"
)

(
    customer_dimension_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(customer_dimension_table)
)

display(
    spark.table(customer_dimension_table)
    .orderBy("user_id")
)

In [0]:
#Define the enriched stream output
stream_static_table = (
    "workspace.stream_join_lab."
    "orders_enriched_stream_static"
)

stream_static_checkpoint = (
    f"{checkpoint_base}/orders_enriched_stream_static"
)

In [0]:
spark.sql(
    f"DROP TABLE IF EXISTS {stream_static_table}"
)

dbutils.fs.rm(
    stream_static_checkpoint,
    True
)

print("✅ Stream-static output reset")

In [0]:
#Read the static dimension
customer_static_df = (
    spark.read
    .table(customer_dimension_table)
)

print(
    f"Orders streaming: {orders_stream_df.isStreaming}"
)

print(
    f"Customer dimension streaming: "
    f"{customer_static_df.isStreaming}"
)

In [0]:
#Join the order stream to the static dimension
orders_enriched_df = (
    orders_stream_df.alias("orders")
    .join(
        customer_static_df.alias("customers"),
        on=F.col("orders.user_id")
        == F.col("customers.user_id"),
        how="left",
    )
    .select(
        F.col("orders.order_event_id"),
        F.col("orders.order_id"),
        F.col("orders.user_id"),
        F.col("orders.order_amount"),
        F.col("orders.order_status"),
        F.col("orders.order_ts"),

        F.coalesce(
            F.col("customers.customer_name"),
            F.lit("Unknown Customer"),
        ).alias("customer_name"),

        F.coalesce(
            F.col("customers.customer_segment"),
            F.lit("UNKNOWN"),
        ).alias("customer_segment"),

        F.coalesce(
            F.col("customers.region"),
            F.lit("UNKNOWN"),
        ).alias("region"),

        F.coalesce(
            F.col("customers.risk_level"),
            F.lit("UNKNOWN"),
        ).alias("risk_level"),

        F.when(
            F.col("customers.user_id").isNotNull(),
            F.lit("MATCHED"),
        )
        .otherwise(
            F.lit("UNKNOWN_CUSTOMER")
        )
        .alias("dimension_match_status"),
    )
)

In [0]:
# Write the enriched order stream
stream_static_query = (
    orders_enriched_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        stream_static_checkpoint,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        stream_static_table
    )
)

stream_static_query.awaitTermination()

print("✅ Stream-static join completed")

In [0]:
%sql
SELECT
    order_id,
    user_id,
    customer_name,
    customer_segment,
    region,
    risk_level,
    order_amount,
    order_ts,
    dimension_match_status
FROM workspace.stream_join_lab.orders_enriched_stream_static
ORDER BY order_ts;

## Stream-Static Join — Result

The order stream was successfully enriched with customer attributes from a static Delta table.

The first three orders matched known customers.

A later order for `U-999` did not have a matching customer record, but the order was preserved through the left join and mapped to the unknown-member values.

### Key takeaway

A stream-static join is best for real-time enrichment when one side is continuously changing and the other side is stable reference data.

It is simpler than a stream-stream join because Spark does not need to retain unmatched records from both sides while waiting for future events.

In [0]:
#Validate the stream-static join
stream_static_result_df = spark.table(
    stream_static_table
)

result_count = (
    stream_static_result_df.count()
)

matched_count = (
    stream_static_result_df
    .filter(
        F.col("dimension_match_status")
        == "MATCHED"
    )
    .count()
)

unknown_count = (
    stream_static_result_df
    .filter(
        F.col("dimension_match_status")
        == "UNKNOWN_CUSTOMER"
    )
    .count()
)

assert result_count == 3, (
    f"Expected 3 enriched orders, found {result_count}"
)

assert matched_count == 3, (
    f"Expected 3 matched customers, found {matched_count}"
)

assert unknown_count == 0, (
    f"Expected no unknown customers, found {unknown_count}"
)

print("✅ Three order events enriched")
print("✅ All customers matched")
print("✅ No streaming state required for the join")
print("✅ Stream-static validation passed")

In [0]:
#Test an unkown customer
orders_batch_2 = [
    Row(
        order_event_id="OE-104",
        order_id="ORD-104",
        user_id="U-999",
        order_amount=65.0,
        order_status="CREATED",
        order_ts=datetime(
            2026, 7, 28, 10, 15, 0
        ),
    ),
]

In [0]:
orders_batch_2_df = spark.createDataFrame(
    orders_batch_2,
    schema=order_schema,
)

(
    orders_batch_2_df
    .coalesce(1)
    .write
    .mode("append")
    .json(
        f"{orders_source_path}/batch_2"
    )
)

display(orders_batch_2_df)

In [0]:
stream_static_query = (
    orders_enriched_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        stream_static_checkpoint,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        stream_static_table
    )
)

stream_static_query.awaitTermination()

print("✅ Orders Batch 2 enriched")

In [0]:
%sql
SELECT
    order_id,
    user_id,
    customer_name,
    customer_segment,
    region,
    dimension_match_status
FROM workspace.stream_join_lab.orders_enriched_stream_static
ORDER BY order_ts;

In [0]:
unknown_order = (
    spark.table(stream_static_table)
    .filter(
        F.col("order_id") == "ORD-104"
    )
    .first()
)

assert unknown_order is not None
assert (
    unknown_order["dimension_match_status"]
    == "UNKNOWN_CUSTOMER"
)
assert (
    unknown_order["customer_name"]
    == "Unknown Customer"
)

print("✅ Unknown customer preserved")
print("✅ Left join prevented order loss")
print("✅ Unknown-member handling passed")

# Part 2 — Stream-Stream Join

## Objective

Correlate two continuously changing streams:

- Order events
- Payment events

The flow is:

Order stream  
+ Payment stream  
→ Watermarks on both sides  
→ Order ID match  
→ Event-time range condition  
→ Matched order-payment events

## Why this join is stateful

Both sides are streaming.

An order may arrive before its payment, and a payment may arrive before the corresponding order is processed.

Spark must retain unmatched records from both streams while waiting for a possible future match.

To control that state, the join uses:

- A watermark on the order stream
- A watermark on the payment stream
- A bounded event-time condition

## Join rule

A payment is considered valid when:

- The order IDs match
- The payment occurs at or after the order
- The payment occurs no more than 10 minutes after the order

This allows Spark to eventually remove records that can no longer produce a valid match.

In [0]:
#Confirm current source data
display(
    spark.read
    .schema(order_schema)
    .option("recursiveFileLookup", "true")
    .json(orders_source_path)
    .orderBy("order_ts")
)

display(
    spark.read
    .schema(payment_schema)
    .option("recursiveFileLookup", "true")
    .json(payments_source_path)
    .orderBy("payment_ts")
)

In [0]:
#add watermark to both streams
orders_with_watermark_df = (
    orders_stream_df
    .withWatermark(
        "order_ts",
        "10 minutes"
    )
)

payments_with_watermark_df = (
    payments_stream_df
    .withWatermark(
        "payment_ts",
        "10 minutes"
    )
)

In [0]:
#Build the time bounded inner join
inner_join_condition = (
    (
        F.col("orders.order_id")
        == F.col("payments.order_id")
    )
    &
    (
        F.col("payments.payment_ts")
        >= F.col("orders.order_ts")
    )
    &
    (
        F.col("payments.payment_ts")
        <= F.col("orders.order_ts")
        + F.expr("INTERVAL 10 MINUTES")
    )
)

In [0]:
#Create joined dataframe
order_payment_inner_df = (
    orders_with_watermark_df.alias("orders")
    .join(
        payments_with_watermark_df.alias("payments"),
        inner_join_condition,
        "inner",
    )
    .select(
        F.col("orders.order_event_id"),
        F.col("orders.order_id"),
        F.col("orders.user_id"),
        F.col("orders.order_amount"),
        F.col("orders.order_status"),
        F.col("orders.order_ts"),

        F.col("payments.payment_event_id"),
        F.col("payments.payment_id"),
        F.col("payments.payment_amount"),
        F.col("payments.payment_status"),
        F.col("payments.payment_ts"),

        (
            F.col("payments.payment_ts").cast("long")
            - F.col("orders.order_ts").cast("long")
        ).alias("payment_delay_seconds"),

        F.when(
            F.col("orders.order_amount")
            == F.col("payments.payment_amount"),
            F.lit("AMOUNT_MATCHED"),
        )
        .otherwise(
            F.lit("AMOUNT_MISMATCH")
        )
        .alias("amount_validation_status"),
    )
)

In [0]:
#reset the inner join output
spark.sql(
    f"DROP TABLE IF EXISTS {inner_join_table}"
)

dbutils.fs.rm(
    inner_join_checkpoint,
    True
)

print("✅ Inner join output reset")

In [0]:
#run the inner stream-stream join
inner_join_query = (
    order_payment_inner_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        inner_join_checkpoint,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        inner_join_table
    )
)

inner_join_query.awaitTermination()

print("✅ Initial stream-stream inner join completed")

In [0]:
%sql
--inspect the initial matches
SELECT
    order_id,
    payment_id,
    user_id,
    order_amount,
    payment_amount,
    order_ts,
    payment_ts,
    payment_delay_seconds,
    amount_validation_status
FROM workspace.stream_join_lab.order_payment_inner_join
ORDER BY order_ts;

In [0]:
#Validate initial inner join
inner_result_df = spark.table(
    inner_join_table
)

initial_match_count = (
    inner_result_df.count()
)

ord_101_count = (
    inner_result_df
    .filter(F.col("order_id") == "ORD-101")
    .count()
)

ord_103_count = (
    inner_result_df
    .filter(F.col("order_id") == "ORD-103")
    .count()
)

ord_102_count = (
    inner_result_df
    .filter(F.col("order_id") == "ORD-102")
    .count()
)

assert initial_match_count == 2, (
    f"Expected 2 matches, found {initial_match_count}"
)

assert ord_101_count == 1
assert ord_103_count == 1
assert ord_102_count == 0

print("✅ ORD-101 matched PAY-101")
print("✅ ORD-103 matched PAY-103")
print("✅ Unpaid ORD-102 excluded from inner join")
print("✅ Initial stream-stream join passed")

In [0]:
#Create payment batch 2
payments_batch_2 = [
    Row(
        payment_event_id="PE-102",
        payment_id="PAY-102",
        order_id="ORD-102",
        payment_amount=80.0,
        payment_status="SUCCESS",
        payment_ts=datetime(
            2026, 7, 28, 10, 12, 0
        ),
    ),

    Row(
        payment_event_id="PE-999",
        payment_id="PAY-999",
        order_id="ORD-999",
        payment_amount=50.0,
        payment_status="SUCCESS",
        payment_ts=datetime(
            2026, 7, 28, 10, 13, 0
        ),
    ),
]

In [0]:
payments_batch_2_df = spark.createDataFrame(
    payments_batch_2,
    schema=payment_schema,
)

(
    payments_batch_2_df
    .coalesce(1)
    .write
    .mode("append")
    .json(
        f"{payments_source_path}/batch_2"
    )
)

display(
    payments_batch_2_df.orderBy("payment_ts")
)

In [0]:
#rerun the inner join with the same checkpoint
inner_join_query = (
    order_payment_inner_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        inner_join_checkpoint,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        inner_join_table
    )
)

inner_join_query.awaitTermination()

print("✅ Payments Batch 2 processed")

In [0]:
%sql
--inspect updated result
SELECT
    order_id,
    payment_id,
    order_amount,
    payment_amount,
    order_ts,
    payment_ts,
    payment_delay_seconds,
    amount_validation_status
FROM workspace.stream_join_lab.order_payment_inner_join
ORDER BY order_ts;

In [0]:
#Validate delayed matching
updated_inner_df = spark.table(
    inner_join_table
)

final_match_count = updated_inner_df.count()

ord_102_match = (
    updated_inner_df
    .filter(
        F.col("order_id") == "ORD-102"
    )
    .first()
)

pay_999_count = (
    updated_inner_df
    .filter(
        F.col("payment_id") == "PAY-999"
    )
    .count()
)

assert final_match_count == 3, (
    f"Expected 3 matches, found {final_match_count}"
)

assert ord_102_match is not None, (
    "Delayed PAY-102 did not match ORD-102"
)

assert (
    ord_102_match["payment_delay_seconds"]
    == 540
), (
    "Expected a 540-second payment delay"
)

assert pay_999_count == 0, (
    "Unknown PAY-999 should not appear "
    "in the inner join"
)

print("✅ Delayed PAY-102 matched ORD-102")
print("✅ Match occurred across micro-batches")
print("✅ PAY-999 remained unmatched")
print("✅ Time-bounded inner join validation passed")

In [0]:
#Add out of range payment
payments_batch_3 = [
    Row(
        payment_event_id="PE-104-LATE",
        payment_id="PAY-104-LATE",
        order_id="ORD-104",
        payment_amount=65.0,
        payment_status="SUCCESS",
        payment_ts=datetime(
            2026, 7, 28, 10, 30, 0
        ),
    )
]

In [0]:
payments_batch_3_df = spark.createDataFrame(
    payments_batch_3,
    schema=payment_schema,
)

(
    payments_batch_3_df
    .coalesce(1)
    .write
    .mode("append")
    .json(
        f"{payments_source_path}/batch_3"
    )
)

display(payments_batch_3_df)

In [0]:
#rerun the inner join
inner_join_query = (
    order_payment_inner_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        inner_join_checkpoint,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        inner_join_table
    )
)

inner_join_query.awaitTermination()

print("✅ Out-of-range payment processed")

In [0]:
#Validate the join
late_payment_count = (
    spark.table(inner_join_table)
    .filter(
        F.col("payment_id")
        == "PAY-104-LATE"
    )
    .count()
)

assert late_payment_count == 0, (
    "PAY-104-LATE should not match because "
    "it arrived outside the 10-minute range"
)

print("✅ Correct order ID was not enough")
print("✅ Time-range condition rejected PAY-104-LATE")

## Why the out-of-range payment did not match

The order ID matched, but the payment occurred 15 minutes after the order.

The join accepts payments only when:

payment_ts >= order_ts

and

payment_ts <= order_ts + 10 minutes

This prevents unrelated or excessively delayed events from being correlated simply because they share the same business key.

# Part 3 — Left Outer Stream-Stream Join

## Objective

Preserve every order, including orders that never receive a valid payment.

A left outer stream-stream join must wait until Spark is confident that no future payment can still match the order.

That means unmatched orders do not appear immediately.

They are emitted only after the relevant payment-side watermark passes the order’s allowed matching window.

In [0]:
#Define left outer output
left_join_table = (
    "workspace.stream_join_lab."
    "order_payment_left_join"
)

left_join_checkpoint = (
    f"{checkpoint_base}/order_payment_left_join"
)

In [0]:
#reset only this new output
spark.sql(
    f"DROP TABLE IF EXISTS {left_join_table}"
)

dbutils.fs.rm(
    left_join_checkpoint,
    True
)

print("✅ Left outer join output reset")

In [0]:
#Build left outer join
order_payment_left_df = (
    orders_with_watermark_df.alias("orders")
    .join(
        payments_with_watermark_df.alias("payments"),
        inner_join_condition,
        "leftOuter",
    )
    .select(
        F.col("orders.order_event_id"),
        F.col("orders.order_id"),
        F.col("orders.user_id"),
        F.col("orders.order_amount"),
        F.col("orders.order_status"),
        F.col("orders.order_ts"),

        F.col("payments.payment_event_id"),
        F.col("payments.payment_id"),
        F.col("payments.payment_amount"),
        F.col("payments.payment_status"),
        F.col("payments.payment_ts"),

        F.when(
            F.col("payments.payment_id").isNotNull(),
            F.lit("PAYMENT_MATCHED"),
        )
        .otherwise(
            F.lit("NO_VALID_PAYMENT")
        )
        .alias("payment_match_status"),

        F.when(
            F.col("payments.payment_ts").isNotNull(),
            (
                F.col("payments.payment_ts").cast("long")
                - F.col("orders.order_ts").cast("long")
            ),
        ).alias("payment_delay_seconds"),
    )
)

In [0]:
#Run the left outer join
left_join_query = (
    order_payment_left_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        left_join_checkpoint,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        left_join_table
    )
)

left_join_query.awaitTermination()

print("✅ Initial left outer join completed")

In [0]:
%sql
--inspect the left join
SELECT
    order_id,
    payment_id,
    order_ts,
    payment_ts,
    payment_match_status
FROM workspace.stream_join_lab.order_payment_left_join
ORDER BY order_ts;

In [0]:
#Advance both watermarks
orders_batch_3 = [
    Row(
        order_event_id="OE-200",
        order_id="ORD-200",
        user_id="U-200",
        order_amount=40.0,
        order_status="CREATED",
        order_ts=datetime(
            2026, 7, 28, 11, 0, 0
        ),
    )
]

payments_batch_4 = [
    Row(
        payment_event_id="PE-200",
        payment_id="PAY-200",
        order_id="ORD-200",
        payment_amount=40.0,
        payment_status="SUCCESS",
        payment_ts=datetime(
            2026, 7, 28, 11, 1, 0
        ),
    )
]

In [0]:
#Write Batch 3
orders_batch_3_df = spark.createDataFrame(
    orders_batch_3,
    schema=order_schema,
)

payments_batch_4_df = spark.createDataFrame(
    payments_batch_4,
    schema=payment_schema,
)

(
    orders_batch_3_df
    .coalesce(1)
    .write
    .mode("append")
    .json(
        f"{orders_source_path}/batch_3"
    )
)

(
    payments_batch_4_df
    .coalesce(1)
    .write
    .mode("append")
    .json(
        f"{payments_source_path}/batch_4"
    )
)

print("✅ Future events written")

In [0]:
#Rerun the left outer join
left_join_query = (
    order_payment_left_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        left_join_checkpoint,
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        left_join_table
    )
)

left_join_query.awaitTermination()

print("✅ Watermarks advanced")

In [0]:
%sql
--Inspect final results
SELECT
    order_id,
    payment_id,
    order_ts,
    payment_ts,
    payment_match_status,
    payment_delay_seconds
FROM workspace.stream_join_lab.order_payment_left_join
ORDER BY order_ts;

In [0]:
#Validate the outer join
left_result_df = spark.table(
    left_join_table
)

ord_104_rows = (
    left_result_df
    .filter(F.col("order_id") == "ORD-104")
    .collect()
)

ord_200_match = (
    left_result_df
    .filter(
        (F.col("order_id") == "ORD-200")
        &
        (
            F.col("payment_match_status")
            == "PAYMENT_MATCHED"
        )
    )
    .count()
)

assert len(ord_104_rows) == 1, (
    f"Expected one ORD-104 result, "
    f"found {len(ord_104_rows)}"
)

assert (
    ord_104_rows[0]["payment_match_status"]
    == "NO_VALID_PAYMENT"
), (
    "ORD-104 should be finalized without "
    "a valid payment"
)

assert ord_200_match == 1, (
    "ORD-200 should match PAY-200"
)

print("✅ Valid payments matched")
print("✅ ORD-104 finalized as unpaid")
print("✅ PAY-104-LATE remained outside the join")
print("✅ Left outer stream-stream join passed")

# Part 4 — State and Runtime Metrics

## Objective

Inspect how Spark manages state for the stream-stream join.

A stream-static join does not need to retain unmatched records from both sides.

A stream-stream join does.

For the stream-stream join, Spark must temporarily retain:

- Orders waiting for payments
- Payments waiting for orders
- Event-time boundaries
- Watermark progress
- Join-state cleanup information

The time-bounded join condition and watermarks allow Spark to eventually remove records that can no longer match.

In [0]:
#Create safe helper function
def safe_int(value, default=0):
    if value is None:
        return default

    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def safe_str(value, default="N/A"):
    if value is None:
        return default

    return str(value)

In [0]:
#Inspect recent progress for the inner join
inner_progress_rows = []

for progress in inner_join_query.recentProgress:
    if not progress:
        continue

    event_time = progress.get("eventTime") or {}
    state_operators = progress.get("stateOperators") or []

    if state_operators:
        state = state_operators[0]
    else:
        state = {}

    inner_progress_rows.append({
        "batch_id": safe_int(
            progress.get("batchId")
        ),
        "input_rows": safe_int(
            progress.get("numInputRows")
        ),
        "watermark": safe_str(
            event_time.get("watermark")
        ),
        "max_event_time": safe_str(
            event_time.get("max")
        ),
        "state_rows_total": safe_int(
            state.get("numRowsTotal")
        ),
        "state_rows_updated": safe_int(
            state.get("numRowsUpdated")
        ),
        "state_rows_removed": safe_int(
            state.get("numRowsRemoved")
        ),
        "rows_dropped_by_watermark": safe_int(
            state.get("numRowsDroppedByWatermark")
        ),
        "state_memory_bytes": safe_int(
            state.get("memoryUsedBytes")
        ),
    })

In [0]:
#define schema
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType,
)

join_progress_schema = StructType([
    StructField("batch_id", LongType(), False),
    StructField("input_rows", LongType(), False),
    StructField("watermark", StringType(), False),
    StructField("max_event_time", StringType(), False),
    StructField("state_rows_total", LongType(), False),
    StructField("state_rows_updated", LongType(), False),
    StructField("state_rows_removed", LongType(), False),
    StructField(
        "rows_dropped_by_watermark",
        LongType(),
        False,
    ),
    StructField(
        "state_memory_bytes",
        LongType(),
        False,
    ),
])

In [0]:
if not inner_progress_rows:
    print("No recent inner-join progress found")
else:
    inner_progress_df = spark.createDataFrame(
        inner_progress_rows,
        schema=join_progress_schema,
    )

    display(
        inner_progress_df.orderBy("batch_id")
    )

In [0]:
#Inspect Left outer join metrics
left_progress_rows = []

for progress in left_join_query.recentProgress:
    if not progress:
        continue

    event_time = progress.get("eventTime") or {}
    state_operators = progress.get("stateOperators") or []

    if state_operators:
        state = state_operators[0]
    else:
        state = {}

    left_progress_rows.append({
        "batch_id": safe_int(
            progress.get("batchId")
        ),
        "input_rows": safe_int(
            progress.get("numInputRows")
        ),
        "watermark": safe_str(
            event_time.get("watermark")
        ),
        "max_event_time": safe_str(
            event_time.get("max")
        ),
        "state_rows_total": safe_int(
            state.get("numRowsTotal")
        ),
        "state_rows_updated": safe_int(
            state.get("numRowsUpdated")
        ),
        "state_rows_removed": safe_int(
            state.get("numRowsRemoved")
        ),
        "rows_dropped_by_watermark": safe_int(
            state.get("numRowsDroppedByWatermark")
        ),
        "state_memory_bytes": safe_int(
            state.get("memoryUsedBytes")
        ),
    })

In [0]:
if not left_progress_rows:
    print("No recent left-join progress found")
else:
    left_progress_df = spark.createDataFrame(
        left_progress_rows,
        schema=join_progress_schema,
    )

    display(
        left_progress_df.orderBy("batch_id")
    )

# Part 5 — Stream-Static vs Stream-Stream

## Stream-static join

One side is streaming and the other side is stable reference data.

Typical purpose:

- Enrichment
- Classification
- Lookup
- Adding descriptive attributes

Characteristics:

- Simpler join
- No watermark required on the static side
- No event-time range condition required
- No need to retain unmatched records from both sides
- Suitable for customer, product, merchant, asset, or location reference data

## Stream-stream join

Both sides continuously change.

Typical purpose:

- Correlating orders with payments
- Joining clicks with conversions
- Matching telemetry with alerts
- Connecting shipment events with delivery confirmations

Characteristics:

- Stateful
- Watermarks required for bounded state
- Time-range condition required
- Late matches can occur across micro-batches
- Unmatched outer-join rows appear only after watermark finalization

In [0]:
#Create the comparison result
comparison_data = [
    (
        "STREAM_STATIC",
        "Orders + customer dimension",
        "One streaming side",
        "No",
        "No",
        "Simple enrichment",
        "Unmatched rows handled immediately",
    ),
    (
        "STREAM_STREAM",
        "Orders + payments",
        "Both sides streaming",
        "Yes",
        "Yes",
        "Stateful event correlation",
        "Unmatched rows wait for watermark",
    ),
]

In [0]:
comparison_schema = StructType([
    StructField("join_type", StringType(), False),
    StructField("example", StringType(), False),
    StructField("changing_sides", StringType(), False),
    StructField("watermark_required", StringType(), False),
    StructField("time_bound_required", StringType(), False),
    StructField("primary_use", StringType(), False),
    StructField("unmatched_behavior", StringType(), False),
])

In [0]:
join_comparison_df = spark.createDataFrame(
    comparison_data,
    schema=comparison_schema,
)

display(join_comparison_df)

In [0]:
%sql
--Final result summary
SELECT
    'STREAM_STATIC' AS join_type,
    COUNT(*) AS output_rows,
    SUM(
        CASE
            WHEN dimension_match_status = 'MATCHED'
            THEN 1
            ELSE 0
        END
    ) AS matched_rows,
    SUM(
        CASE
            WHEN dimension_match_status = 'UNKNOWN_CUSTOMER'
            THEN 1
            ELSE 0
        END
    ) AS unmatched_rows
FROM workspace.stream_join_lab.orders_enriched_stream_static

UNION ALL

SELECT
    'STREAM_STREAM_INNER' AS join_type,
    COUNT(*) AS output_rows,
    COUNT(*) AS matched_rows,
    0 AS unmatched_rows
FROM workspace.stream_join_lab.order_payment_inner_join

UNION ALL

SELECT
    'STREAM_STREAM_LEFT' AS join_type,
    COUNT(*) AS output_rows,
    SUM(
        CASE
            WHEN payment_match_status = 'PAYMENT_MATCHED'
            THEN 1
            ELSE 0
        END
    ) AS matched_rows,
    SUM(
        CASE
            WHEN payment_match_status = 'NO_VALID_PAYMENT'
            THEN 1
            ELSE 0
        END
    ) AS unmatched_rows
FROM workspace.stream_join_lab.order_payment_left_join;

In [0]:
#Final Validation
stream_static_df = spark.table(
    stream_static_table
)

inner_join_df = spark.table(
    inner_join_table
)

left_join_df = spark.table(
    left_join_table
)

assert stream_static_df.count() == 4, (
    "Expected 4 stream-static results"
)

assert (
    stream_static_df
    .filter(
        F.col("dimension_match_status")
        == "UNKNOWN_CUSTOMER"
    )
    .count()
    == 1
), "Expected one unknown customer"

assert inner_join_df.count() >= 3, (
    "Expected at least three valid inner-join matches"
)

assert (
    inner_join_df
    .filter(
        F.col("payment_id")
        == "PAY-104-LATE"
    )
    .count()
    == 0
), "Out-of-range payment should not match"

assert (
    left_join_df
    .filter(
        (F.col("order_id") == "ORD-104")
        &
        (
            F.col("payment_match_status")
            == "NO_VALID_PAYMENT"
        )
    )
    .count()
    == 1
), "ORD-104 should be finalized without payment"

print("✅ Stream-static enrichment passed")
print("✅ Unknown customer preserved")
print("✅ Stream-stream delayed match passed")
print("✅ Out-of-range payment rejected")
print("✅ Unpaid order finalized after watermark")
print("✅ Post 3 implementation complete")